### Import library and check CUDA

In [2]:
%pip install datasets
%pip install evaluate

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from dataclasses import dataclass
from datasets import load_dataset, concatenate_datasets, ClassLabel, Dataset
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder
import torch
import evaluate
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

c:\Anaconda3\envs\willi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
if torch.cuda.is_available():
    print(f"CUDA is available! Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. Running on CPU.")

CUDA is available! Using GPU: NVIDIA GeForce RTX 4060 Ti


### Dataset and Training Script

#### Process dataset from raw csv (From Kaggle)

In [5]:
df = pd.read_csv("./data.csv")
df.head()

,Sl no,Tweets,Search key,Feeling
0,1,"#1: @fe ed ""RT @MirayaDizon1: Time is ticking...",happy moments,happy
1,2,"#2: @蓮花 &はすか ed ""RT @ninjaryugo: ＃コナモンの日 だそうで...",happy moments,happy
2,3,"#3: @Ris ♡ ed ""Happy birthday to one smokin h...",happy moments,happy
3,4,"#4: @월월 [씍쯴사랑로봇] jwinnie is the best, cheer u...",happy moments,happy
4,5,"#5: @Madhurima wth u vc♥ ed ""Good morning dea...",happy moments,happy


In [6]:
df.columns

Index(['Sl no', 'Tweets', 'Search key', 'Feeling'], dtype='object')

In [7]:
df['label'] = df['Feeling']
df['text'] = df['Tweets']

df.drop(columns=['Tweets', 'Sl no', 'Search key', 'Feeling'], inplace=True)

In [8]:
df.head()

,label,text
0,happy,"#1: @fe ed ""RT @MirayaDizon1: Time is ticking..."
1,happy,"#2: @蓮花 &はすか ed ""RT @ninjaryugo: ＃コナモンの日 だそうで..."
2,happy,"#3: @Ris ♡ ed ""Happy birthday to one smokin h..."
3,happy,"#4: @월월 [씍쯴사랑로봇] jwinnie is the best, cheer u..."
4,happy,"#5: @Madhurima wth u vc♥ ed ""Good morning dea..."


In [9]:
df.groupby('label').count()

,text
label,
angry,1341
disgust,637
fear,863
happy,3928
sad,2849
surprise,399


In [10]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df['label'])

In [11]:
# Get label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print("Label Mapping:", label_mapping)

Label Mapping: {'angry': np.int64(0), 'disgust': np.int64(1), 'fear': np.int64(2), 'happy': np.int64(3), 'sad': np.int64(4), 'surprise': np.int64(5)}


In [12]:
df.head()

,label,text
0,3,"#1: @fe ed ""RT @MirayaDizon1: Time is ticking..."
1,3,"#2: @蓮花 &はすか ed ""RT @ninjaryugo: ＃コナモンの日 だそうで..."
2,3,"#3: @Ris ♡ ed ""Happy birthday to one smokin h..."
3,3,"#4: @월월 [씍쯴사랑로봇] jwinnie is the best, cheer u..."
4,3,"#5: @Madhurima wth u vc♥ ed ""Good morning dea..."


In [13]:
df = df.sample(frac=1).reset_index(drop=True)
df.head()

,label,text
0,3,"#66: @シイヤ(猫背) ed ""【自動】規制垢→( shy_a4i_2nd shy_a..."
1,3,"#8: @StaceyC ed ""RT @Monaheart1229: Just now ..."
2,0,"#80: @PSNI RBLX ed ""RT @ThomasB84091325: SUPT..."
3,3,"#20: @Dubs ed ""@BrandonDavisBD @Russo_Brother..."
4,3,"#37: @L Y : A N N E ed ""RT @YOTTO_R: Learn to..."


In [14]:
df.to_csv("process_data.csv")

#### Process CSV for Indonesia Dataset

In [15]:
df_indo = pd.read_csv('./PRDECT-ID Dataset.csv')
df_indo.head()

,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,Computers and Laptops,SSD Midasforce 128 Gb - Tanpa Caddy,Jakarta Barat,213000,5.0,12300,3573,5,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [16]:
df_indo.columns

Index(['Category', 'Product Name', 'Location', 'Price', 'Overall Rating',
       'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review',
       'Sentiment', 'Emotion'],
      dtype='object')

In [17]:
drop_columns = ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Sentiment', 'Customer Review', 'Emotion']

df_indo['text'] = df_indo['Customer Review']
df_indo['label'] = df_indo['Emotion']
df_indo.drop(columns=drop_columns, inplace=True)
df_indo.head()

,text,label
0,Alhamdulillah berfungsi dengan baik. Packaging...,Happy
1,"barang bagus dan respon cepat, harga bersaing ...",Happy
2,"barang bagus, berfungsi dengan baik, seler ram...",Happy
3,bagus sesuai harapan penjual nya juga ramah. t...,Happy
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Happy


In [18]:
df_indo['label'] = label_encoder.fit_transform(df_indo['label'])

In [19]:
df_indo.head()

,text,label
0,Alhamdulillah berfungsi dengan baik. Packaging...,2
1,"barang bagus dan respon cepat, harga bersaing ...",2
2,"barang bagus, berfungsi dengan baik, seler ram...",2
3,bagus sesuai harapan penjual nya juga ramah. t...,2
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",2


In [20]:
df_indo = df_indo.sample(frac=1).reset_index(drop=True)
df_indo.to_csv("indo-data-review.csv")

#### Constants, that will be used for training, loading, process, tokenizing data.

In [21]:
DISTIL_BERT = "distilbert-base-uncased"
ROBERTA_BASE = "roberta-base"
XLM_ROBERTA_BASE = "xlm-roberta-base"
BERT_BASE = "google-bert/bert-base-uncased"
INDO_BERT_BASE = "indobenchmark/indobert-base-p1"

DATASETS_LINKS: list[str] = [
    'AdamCodd/emotion-balanced',
    'dair-ai/emotion',
    # 'philschmid/emotion',
    'SetFit/emotion',
    'mteb/emotion'
]

KAGGLE_DATASET = "process_data.csv"
INDO_DATASET = "./indo-data-review.csv"

#### Data classes that is going to served as an interfaces for the classes

In [22]:
@dataclass
class DataLoaderSettings:
    dataset_link: str
    keys: list[str]
    text_col: str
    label_col: str
    hf_dataset: bool = True

@dataclass
class DatasetSettings:
    label_col: str
    text_col: str
    tokenizer_link: str

@dataclass
class TrainingInformation:
    pretrained_model: str

#### Data Loader, will load the data, process (tokenize, remove stop words and other important things), convert labels to class label.

In [23]:
class DataLoader:
    def __init__(self, loader_settings: DataLoaderSettings):
        self.stop_words = stopwords.words('english')
        self.settings = loader_settings
        self.loaded = self.load_dataset(loader_settings.hf_dataset)
        self.processed = self.process()
        self.convert_labels_to_classlabel()

        print(f'Loaded: {self.loaded}')
        print(f'Processed: {self.processed}')

    def load_dataset(self, hf_dataset=True) -> bool:
        if not hf_dataset:
            print("KAGGLE")
            try:
                df = pd.read_csv(self.settings.dataset_link)
                self.dataset = Dataset.from_pandas(df)
                return True
            except Exception as e:
                print(f"Loading dataset went wrong: {e}")
                return False

        try:
            dataset = load_dataset(self.settings.dataset_link, trust_remote_code=True)
            merged_dataset = None

            if not isinstance(dataset, dict):
                self.dataset = dataset
                return True

            for key in self.settings.keys:
                if key not in dataset:
                    continue

                dataset_partition = dataset[key]

                if merged_dataset == None:
                    merged_dataset = dataset_partition
                else:
                    merged_dataset = concatenate_datasets([merged_dataset, dataset_partition])

            self.dataset = merged_dataset
            return True
        except Exception as e:
            print(f'Something went wrong when trying to load dataset from link {self.settings.dataset_link}')
            print(f'Got error {e}')
            return False

    def convert_labels_to_classlabel(self):
        unique_labels = list(set(self.dataset[self.settings.label_col]))
        class_label_feature = ClassLabel(num_classes=len(unique_labels), names=[str(label) for label in unique_labels])

        self.dataset = self.dataset.map(lambda example: {self.settings.label_col: class_label_feature.str2int(str(example[self.settings.label_col]))})
        self.dataset = self.dataset.cast_column(self.settings.label_col, class_label_feature)
        print(type(self.dataset[0]['label']))

    def process(self) -> bool:
        if not self.loaded:
            raise ValueError("Dataset has not been loaded")

        def process_text(sample) -> str:
            text = sample[self.settings.text_col]
            words = self.tokenize(text)
            words = self.remove_stopwords(words)

            sample[self.settings.text_col] = ' '.join(words)
            return sample

        try:
            self.dataset = self.dataset.map(process_text)
            return True
        except Exception as e:
            print(e)
            return False

    def tokenize(self, text) -> list[str]:
        words = word_tokenize(text)
        return words

    def remove_stopwords(self, words) -> list[str]:
        words = [word for word in words if word not in self.stop_words and word.isalpha()]
        return words

#### Custom dataset that is going to be used for splitting dataset, tokenizing dataset.

In [24]:
class CustomDataset:
    """
    Custom dataset class for tokenizing text data.

    Attributes:
    - dataset: DataFrame loaded from CSV
    - tokenizer: Tokenizer for text processing
    - max_length: Maximum token length
    """

    def __init__(self, dataset_settings: DatasetSettings, data_loader: DataLoader, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(dataset_settings.tokenizer_link)
        self.settings = dataset_settings
        self.data_loader = data_loader
        self.max_length = max_length
        self.label_encoder = LabelEncoder()
        self.splitted = self.split_dataset()
        self.tokenized = self.tokenize_datasets()
        print(f'Splitted: {self.splitted}')
        print(f'Tokenized: {self.tokenized}')

    def count_unique_labels(self) -> int:
        """
        Counts the number of unique labels in the dataset.

        Returns:
        - int: The number of unique labels.
        """
        try:
            label_column = self.settings.label_col

            # Get unique labels from dataset
            unique_labels = set(self.train[label_column])

            print(f"Number of unique labels: {len(unique_labels)}")
            return len(unique_labels)

        except Exception as e:
            print(f"Error counting unique labels: {e}")
            return 0

    def __len__(self):
        return len(self.data)

    def split_dataset(self, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42) -> bool:
        """
        Split the dataset into training, validation, and testing sets.

        Args:
        - train_ratio (float): Proportion of the dataset to use for training.
        - val_ratio (float): Proportion of the dataset to use for validation.
        - test_ratio (float): Proportion of the dataset to use for testing.
        - random_state (int): Seed for reproducibility.

        Returns:
        - bool: True if the split was successful, False otherwise.
        """
        if not hasattr(self, "data_loader"):
            print("Dataset is not loaded.")
            return False

        if not (0 < train_ratio < 1 and 0 < val_ratio < 1 and 0 < test_ratio < 1 and train_ratio + val_ratio + test_ratio == 1):
            print("Invalid split ratios. Ensure they sum to 1.")
            return False

        dataset = self.data_loader.dataset
        
        try:
            # First, split into train and temp (val + test)
            train_test_split = dataset.train_test_split(test_size=(1 - train_ratio), seed=seed, stratify_by_column=self.settings.label_col)
            train_data = train_test_split["train"]
            temp_data = train_test_split["test"]

            # Compute relative validation split
            val_size = val_ratio / (val_ratio + test_ratio)  # Normalize val/test split
            val_test_split = temp_data.train_test_split(test_size=(1 - val_size), seed=seed, stratify_by_column=self.settings.label_col)

            self.train = train_data
            self.val = val_test_split["train"]
            self.test = val_test_split["test"]

            print(f"Dataset split complete: Train({len(self.train)}), Val({len(self.val)}), Test({len(self.test)})")
            return True

        except Exception as e:
            print(f"Error splitting dataset: {e}")
            return False

    def tokenize_datasets(self):
        """
        Tokenizes the train, validation, and test datasets using the tokenizer.

        This function modifies self.train, self.val, and self.test in-place.
        """
        if not hasattr(self, "train") or self.train is None:
            print("Training dataset is not loaded.")
            return False
        if not hasattr(self, "val") or self.val is None:
            print("Validation dataset is not loaded.")
            return False
        if not hasattr(self, "test") or self.test is None:
            print("Testing dataset is not loaded.")
            return False

        try:
            text_column = self.settings.text_col
            label_column = self.settings.label_col

            # Tokenization function
            def tokenize_function(example):
                encoding = self.tokenizer(
                    example[text_column],
                    padding="max_length",
                    truncation=True,
                    max_length=self.max_length
                )
                encoding["labels"] = [torch.tensor(label, dtype=torch.long) for label in example[label_column]]
                return encoding

            self.train = self.train.map(tokenize_function, batched=True)
            self.val = self.val.map(tokenize_function, batched=True)
            self.test = self.test.map(tokenize_function, batched=True)

            print("Tokenization complete for train, val, and test datasets.")
            return True

        except Exception as e:
            print(f"Error tokenizing datasets: {e}")
            return False

#### Training script and functions with additional evaluation metrics.

In [25]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

In [26]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    recall = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"]
    }

def train_model(dataset: CustomDataset, training_information: TrainingInformation, epoch=3):
    unique_labels = dataset.count_unique_labels()
    print(f'Unique Label Count: {unique_labels}')
    model = AutoModelForSequenceClassification.from_pretrained(training_information.pretrained_model, num_labels=unique_labels)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=epoch,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset.train,
        eval_dataset=dataset.val,
        compute_metrics=compute_metrics
    )
    trainer.train()

    evaluation_result = trainer.evaluate(dataset.test)
    print("EVALUATION RESULT")
    print(evaluation_result)

### Training From Huggin Face Data

#### BERT

In [26]:
bert_dataset: list[CustomDataset] = []

for dataset_link in DATASETS_LINKS:
    print(dataset_link)
    print('')

    loader_settings = DataLoaderSettings(
        dataset_link=dataset_link,
        keys=['train', 'validation', 'test'],
        label_col='label',
        text_col='text'
    )

    loader = DataLoader(loader_settings=loader_settings)
    print(loader.dataset[0])

    dataset_settings = DatasetSettings(
        tokenizer_link=BERT_BASE,
        label_col='label',
        text_col='text'
    )

    dataset = CustomDataset(
        data_loader=loader,
        dataset_settings=dataset_settings
    )

    bert_dataset.append(dataset)

AdamCodd/emotion-balanced

<class 'int'>
Loaded: True
Processed: True
{'text': 'sick feeling like want opinions please nothing rude imature', 'label': 3}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 6252.33 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
dair-ai/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 5939.54 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
SetFit/emotion



...

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 6157.70 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
mteb/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 6261.99 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [27]:
for idx in range(len(bert_dataset)):
    print(DATASETS_LINKS[idx])

    training_information = TrainingInformation(
        pretrained_model=BERT_BASE
    )

    train_model(bert_dataset[idx], training_information=training_information)

AdamCodd/emotion-balanced
Number of unique labels: 6
Unique Label Count: 6


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.222600,0.230594,0.942000,0.943750,0.942059,0.941570
2,0.122300,0.191927,0.945000,0.947428,0.945047,0.945002
3,0.065700,0.218449,0.943000,0.944862,0.943057,0.942738


EVALUATION RESULT
{'eval_loss': 0.13940578699111938, 'eval_accuracy': 0.962, 'eval_precision': 0.963049087200076, 'eval_recall': 0.9619769470068871, 'eval_f1': 0.9618621994589124, 'eval_runtime': 23.3109, 'eval_samples_per_second': 85.797, 'eval_steps_per_second': 10.725, 'epoch': 3.0}
dair-ai/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.169100,0.205459,0.934500,0.890046,0.924017,0.901891
2,0.033000,0.189211,0.932000,0.904268,0.901256,0.897464
3,0.237800,0.210545,0.937500,0.902749,0.913122,0.907435


EVALUATION RESULT
{'eval_loss': 0.18735477328300476, 'eval_accuracy': 0.943, 'eval_precision': 0.9170146987746838, 'eval_recall': 0.918277233905798, 'eval_f1': 0.9175299750226396, 'eval_runtime': 22.4314, 'eval_samples_per_second': 89.161, 'eval_steps_per_second': 11.145, 'epoch': 3.0}
mteb/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.169100,0.205459,0.934500,0.890046,0.924017,0.901891
2,0.033000,0.189211,0.932000,0.904268,0.901256,0.897464
3,0.237800,0.210545,0.937500,0.902749,0.913122,0.907435


EVALUATION RESULT
{'eval_loss': 0.18735474348068237, 'eval_accuracy': 0.943, 'eval_precision': 0.9170146987746838, 'eval_recall': 0.918277233905798, 'eval_f1': 0.9175299750226396, 'eval_runtime': 22.4346, 'eval_samples_per_second': 89.148, 'eval_steps_per_second': 11.143, 'epoch': 3.0}


#### DistilBert

In [10]:
distil_dataset: list[CustomDataset] = []

for dataset_link in DATASETS_LINKS:
    print(dataset_link)
    print('')

    loader_settings = DataLoaderSettings(
        dataset_link=dataset_link,
        keys=['train', 'validation', 'test'],
        label_col='label',
        text_col='text'
    )

    loader = DataLoader(loader_settings=loader_settings)
    print(loader.dataset[0])

    dataset_settings = DatasetSettings(
        tokenizer_link=DISTIL_BERT,
        label_col='label',
        text_col='text'
    )

    dataset = CustomDataset(
        data_loader=loader,
        dataset_settings=dataset_settings
    )

    distil_dataset.append(dataset)

AdamCodd/emotion-balanced

<class 'int'>
Loaded: True
Processed: True
{'text': 'sick feeling like want opinions please nothing rude imature', 'label': 3}
Dataset split complete: Train(16000), Val(2000), Test(2000)
Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
dair-ai/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0}
Dataset split complete: Train(16000), Val(2000), Test(2000)
Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
SetFit/emotion



Repo card metadata block was not found. Setting CardData to empty.


<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)
Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
mteb/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)
Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [11]:
for idx in range(len(distil_dataset)):
    print(DATASETS_LINKS[idx])

    training_information = TrainingInformation(
        pretrained_model=DISTIL_BERT
    )

    train_model(distil_dataset[idx], training_information=training_information)

AdamCodd/emotion-balanced
Number of unique labels: 6
Unique Label Count: 6


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.228700,0.179467,0.952000,0.952823,0.951991,0.951932
2,0.125500,0.132215,0.959000,0.960073,0.958978,0.958827
3,0.074100,0.166610,0.959500,0.960893,0.959473,0.959307


EVALUATION RESULT
{'eval_loss': 0.16660991311073303, 'eval_accuracy': 0.9595, 'eval_precision': 0.9608927311205355, 'eval_recall': 0.9594729459998921, 'eval_f1': 0.9593074596372793, 'eval_runtime': 11.49, 'eval_samples_per_second': 174.064, 'eval_steps_per_second': 21.758, 'epoch': 3.0}
dair-ai/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.252900,0.227385,0.927000,0.890952,0.924740,0.903741
2,0.076500,0.172986,0.937500,0.922579,0.907827,0.910134
3,0.174800,0.221734,0.935000,0.904214,0.905648,0.904718


EVALUATION RESULT
{'eval_loss': 0.22173364460468292, 'eval_accuracy': 0.935, 'eval_precision': 0.9042141941692882, 'eval_recall': 0.9056478978455448, 'eval_f1': 0.904718414562378, 'eval_runtime': 11.6813, 'eval_samples_per_second': 171.214, 'eval_steps_per_second': 21.402, 'epoch': 3.0}
mteb/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.252900,0.227385,0.927000,0.890952,0.924740,0.903741
2,0.076500,0.172986,0.937500,0.922579,0.907827,0.910134
3,0.174800,0.221734,0.935000,0.904214,0.905648,0.904718


EVALUATION RESULT
{'eval_loss': 0.2217336744070053, 'eval_accuracy': 0.935, 'eval_precision': 0.9042141941692882, 'eval_recall': 0.9056478978455448, 'eval_f1': 0.904718414562378, 'eval_runtime': 11.4833, 'eval_samples_per_second': 174.167, 'eval_steps_per_second': 21.771, 'epoch': 3.0}


#### ROBERTA

In [12]:
roberta_dataset: list[CustomDataset] = []

for dataset_link in DATASETS_LINKS:
    print(dataset_link)
    print('')

    loader_settings = DataLoaderSettings(
        dataset_link=dataset_link,
        keys=['train', 'validation', 'test'],
        label_col='label',
        text_col='text'
    )

    loader = DataLoader(loader_settings=loader_settings)
    print(loader.dataset[0])

    dataset_settings = DatasetSettings(
        tokenizer_link=ROBERTA_BASE,
        label_col='label',
        text_col='text'
    )

    dataset = CustomDataset(
        data_loader=loader,
        dataset_settings=dataset_settings
    )

    roberta_dataset.append(dataset)

AdamCodd/emotion-balanced



Casting the dataset: 100%|██████████| 20000/20000 [00:00<00:00, 1666721.24 examples/s]


<class 'int'>
Loaded: True
Processed: True
{'text': 'sick feeling like want opinions please nothing rude imature', 'label': 3}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8929.17 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
dair-ai/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0}
Dataset split complete: Train(16000), Val(2000), Test(2000)


...

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8635.99 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
mteb/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8587.43 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [13]:
for idx in range(len(roberta_dataset)):
    print(DATASETS_LINKS[idx])

    training_information = TrainingInformation( 
        pretrained_model=ROBERTA_BASE
    )

    train_model(roberta_dataset[idx], training_information=training_information)

AdamCodd/emotion-balanced
Number of unique labels: 6
Unique Label Count: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.536400,0.255567,0.941500,0.942801,0.941495,0.941399
2,0.140700,0.178524,0.954000,0.955272,0.953987,0.953930
3,0.062700,0.167556,0.959000,0.959885,0.958981,0.958898


EVALUATION RESULT
{'eval_loss': 0.1675557941198349, 'eval_accuracy': 0.959, 'eval_precision': 0.9598846889364507, 'eval_recall': 0.9589814365263467, 'eval_f1': 0.958898448216419, 'eval_runtime': 22.9509, 'eval_samples_per_second': 87.143, 'eval_steps_per_second': 10.893, 'epoch': 3.0}
dair-ai/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.251200,0.283425,0.925000,0.884038,0.920452,0.898245
2,0.035400,0.199040,0.932500,0.917191,0.906216,0.905748
3,0.130900,0.179232,0.937000,0.925571,0.903496,0.910988


EVALUATION RESULT
{'eval_loss': 0.17923152446746826, 'eval_accuracy': 0.937, 'eval_precision': 0.9255708268333076, 'eval_recall': 0.9034964964323824, 'eval_f1': 0.9109876248475293, 'eval_runtime': 21.3248, 'eval_samples_per_second': 93.788, 'eval_steps_per_second': 11.723, 'epoch': 3.0}
mteb/emotion
Number of unique labels: 6
Unique Label Count: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.251200,0.283425,0.925000,0.884038,0.920452,0.898245
2,0.035400,0.199040,0.932500,0.917191,0.906216,0.905748
3,0.130900,0.179232,0.937000,0.925571,0.903496,0.910988


EVALUATION RESULT
{'eval_loss': 0.17923153936862946, 'eval_accuracy': 0.937, 'eval_precision': 0.9255708268333076, 'eval_recall': 0.9034964964323824, 'eval_f1': 0.9109876248475293, 'eval_runtime': 21.3678, 'eval_samples_per_second': 93.599, 'eval_steps_per_second': 11.7, 'epoch': 3.0}


#### XLM RoBERTa Base

In [10]:
xlm_roberta: list[CustomDataset] = []

for dataset_link in DATASETS_LINKS:
    print(dataset_link)
    print('')

    loader_settings = DataLoaderSettings(
        dataset_link=dataset_link,
        keys=['train', 'validation', 'test'],
        label_col='label',
        text_col='text'
    )

    loader = DataLoader(loader_settings=loader_settings)
    print(loader.dataset[0])

    dataset_settings = DatasetSettings(
        tokenizer_link=XLM_ROBERTA_BASE,
        label_col='label',
        text_col='text'
    )

    dataset = CustomDataset(
        data_loader=loader,
        dataset_settings=dataset_settings
    )

    xlm_roberta.append(dataset)

AdamCodd/emotion-balanced

<class 'int'>
Loaded: True
Processed: True
{'text': 'sick feeling like want opinions please nothing rude imature', 'label': 3}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 7991.67 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
dair-ai/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8547.38 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
SetFit/emotion



...

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8525.44 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
mteb/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 8417.31 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [17]:
torch.cuda.empty_cache()

In [ ]:
# for idx in range(len(xlm_roberta)):
#     print(DATASETS_LINKS[idx])

#     training_information = TrainingInformation(
#         pretrained_model=XLM_ROBERTA_BASE
#     )

#     train_model(xlm_roberta[idx], training_information=training_information)

AdamCodd/emotion-balanced
Number of unique labels: 6
Unique Label Count: 6


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Anaconda3\envs\willi\Lib\site-packages\IPython\core\interactiveshell.py", line 2170, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Anaconda3\envs\willi\Lib\site-packages\IPython\core\ultratb.py", line 1457, in structured_traceback
    return FormattedTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Anaconda3\envs\willi\Lib\site-packages\IPython\core\ultratb.py", line 1348, in structured_traceback
    return VerboseTB.structured_traceback(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Anaconda3\envs\willi\Lib\site-packages\IPython\core\ultratb.py", line 1195, in structured_traceback
    formatted_exception = self.format_exception_as_a_whole(etype, evalue, etb, number_of_lines_of_context,
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Anaconda3\envs\w

### Kaggle Dataset Experiment.

#### DistilBert

In [19]:
kaggle_loader_settings = DataLoaderSettings(
    dataset_link=KAGGLE_DATASET,
    hf_dataset=False,
    keys=[],
    label_col='label',
    text_col='text'
)

kaggle_dataset_setting = DatasetSettings(
    label_col='label',
    text_col='text',
    tokenizer_link=DISTIL_BERT
)

kaggle_loader = DataLoader(loader_settings=kaggle_loader_settings)

kaggle_dataset = CustomDataset(
    data_loader=kaggle_loader,
    dataset_settings=kaggle_dataset_setting,
)

training_information = TrainingInformation(pretrained_model=DISTIL_BERT)

KAGGLE


Casting the dataset: 100%|██████████| 10017/10017 [00:00<00:00, 4991012.49 examples/s]


<class 'int'>
Loaded: True
Processed: True
Dataset split complete: Train(8013), Val(1002), Test(1002)


Map: 100%|██████████| 1002/1002 [00:00<00:00, 7573.22 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [20]:
train_model(kaggle_dataset, training_information)

Number of unique labels: 6
Unique Label Count: 6


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.372800,0.410380,0.876248,0.891853,0.819422,0.851159
2,0.221700,0.397963,0.896208,0.894122,0.860383,0.876098
3,0.236400,0.491056,0.896208,0.891656,0.863432,0.876508


EVALUATION RESULT
{'eval_loss': 0.46213433146476746, 'eval_accuracy': 0.9011976047904192, 'eval_precision': 0.8868972383166449, 'eval_recall': 0.8934844550791077, 'eval_f1': 0.8889257496944896, 'eval_runtime': 5.744, 'eval_samples_per_second': 174.442, 'eval_steps_per_second': 21.936, 'epoch': 3.0}


#### Roberta

In [21]:
kaggle_loader_settings = DataLoaderSettings(
    dataset_link=KAGGLE_DATASET,
    hf_dataset=False,
    keys=[],
    label_col='label',
    text_col='text'
)

kaggle_dataset_setting = DatasetSettings(
    label_col='label',
    text_col='text',
    tokenizer_link=ROBERTA_BASE
)

kaggle_loader = DataLoader(loader_settings=kaggle_loader_settings)

kaggle_dataset = CustomDataset(
    data_loader=kaggle_loader,
    dataset_settings=kaggle_dataset_setting,
)

training_information = TrainingInformation(pretrained_model=ROBERTA_BASE)

KAGGLE


Casting the dataset: 100%|██████████| 10017/10017 [00:00<00:00, 830602.04 examples/s]


<class 'int'>
Loaded: True
Processed: True
Dataset split complete: Train(8013), Val(1002), Test(1002)


Map: 100%|██████████| 1002/1002 [00:00<00:00, 7951.03 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [22]:
train_model(kaggle_dataset, training_information)

Number of unique labels: 6
Unique Label Count: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.412900,0.549299,0.831337,0.787718,0.796011,0.786467
2,0.446900,0.475849,0.879242,0.917173,0.819020,0.857278
3,0.149600,0.542149,0.872255,0.902426,0.827471,0.859284


EVALUATION RESULT
{'eval_loss': 0.46178486943244934, 'eval_accuracy': 0.8972055888223552, 'eval_precision': 0.914345661308306, 'eval_recall': 0.8878873061862854, 'eval_f1': 0.8989235684065989, 'eval_runtime': 10.6812, 'eval_samples_per_second': 93.81, 'eval_steps_per_second': 11.796, 'epoch': 3.0}


### Indonesia Dataset Language

#### IndoBERT

In [27]:
indo_loader_settings = DataLoaderSettings(
    dataset_link=INDO_DATASET,
    hf_dataset=False,
    keys=[],
    label_col='label',
    text_col='text'
)

indo_dataset_setting = DatasetSettings(
    label_col='label',
    text_col='text',
    tokenizer_link=INDO_BERT_BASE
)

indo_loader = DataLoader(loader_settings=indo_loader_settings)

indo_dataset = CustomDataset(
    data_loader=indo_loader,
    dataset_settings=indo_dataset_setting,
)

training_information = TrainingInformation(pretrained_model=INDO_BERT_BASE)

KAGGLE


Casting the dataset: 100%|██████████| 5400/5400 [00:00<00:00, 2612972.04 examples/s]


<class 'int'>
Loaded: True
Processed: True
Dataset split complete: Train(4320), Val(540), Test(540)


Map: 100%|██████████| 540/540 [00:00<00:00, 5981.26 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [28]:
torch.cuda.empty_cache()

In [29]:
train_model(indo_dataset, training_information, epoch=8)

Number of unique labels: 5
Unique Label Count: 5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.750900,0.894465,0.625926,0.632713,0.565744,0.527537
2,0.998400,0.762156,0.670370,0.653535,0.648240,0.640606
3,0.552200,0.809253,0.674074,0.653324,0.650012,0.644758
4,0.368600,1.346096,0.655556,0.621962,0.617647,0.618286
5,0.208900,1.429994,0.688889,0.654949,0.662098,0.655033
6,0.004600,1.636774,0.707407,0.675689,0.670642,0.672906
7,0.004100,1.924894,0.694444,0.667026,0.645829,0.652372
8,0.084100,1.877285,0.696296,0.659991,0.661655,0.660451


EVALUATION RESULT
{'eval_loss': 2.0909550189971924, 'eval_accuracy': 0.6537037037037037, 'eval_precision': 0.6106447891958557, 'eval_recall': 0.6111092265722577, 'eval_f1': 0.6056251999989739, 'eval_runtime': 6.0646, 'eval_samples_per_second': 89.042, 'eval_steps_per_second': 11.213, 'epoch': 8.0}


#### BertBase

In [30]:
indo_loader_settings = DataLoaderSettings(
    dataset_link=INDO_DATASET,
    hf_dataset=False,
    keys=[],
    label_col='label',
    text_col='text'
)

indo_dataset_setting = DatasetSettings(
    label_col='label',
    text_col='text',
    tokenizer_link=BERT_BASE
)

indo_loader = DataLoader(loader_settings=indo_loader_settings)

indo_dataset = CustomDataset(
    data_loader=indo_loader,
    dataset_settings=indo_dataset_setting,
)

training_information = TrainingInformation(pretrained_model=BERT_BASE)

KAGGLE


Casting the dataset: 100%|██████████| 5400/5400 [00:00<00:00, 2567359.06 examples/s]


<class 'int'>
Loaded: True
Processed: True
Dataset split complete: Train(4320), Val(540), Test(540)


Map: 100%|██████████| 540/540 [00:00<00:00, 6067.38 examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [31]:
train_model(indo_dataset, training_information, epoch=8)

Number of unique labels: 5
Unique Label Count: 5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.059200,0.962103,0.574074,0.659238,0.473087,0.416885
2,1.071500,0.868985,0.638889,0.638270,0.590515,0.590435
3,0.786300,0.798953,0.642593,0.593338,0.595111,0.587850
4,0.616900,0.869002,0.644444,0.638504,0.608629,0.605036
5,0.416900,1.016954,0.664815,0.626648,0.632070,0.626235
6,0.160900,1.336263,0.653704,0.606686,0.606427,0.602390
7,0.110900,1.695993,0.661111,0.622483,0.617907,0.619838
8,0.169700,1.837891,0.672222,0.634923,0.618911,0.624650


EVALUATION RESULT
{'eval_loss': 1.9816440343856812, 'eval_accuracy': 0.6333333333333333, 'eval_precision': 0.5894085514651779, 'eval_recall': 0.5797196521011309, 'eval_f1': 0.5812391530038589, 'eval_runtime': 6.162, 'eval_samples_per_second': 87.635, 'eval_steps_per_second': 11.035, 'epoch': 8.0}
